# 05. 파이프라인 — 공통 정의는 `src/`로, 실험은 여기서

## 이 노트북의 역할

`04_model_selection.ipynb`가 **231셀**까지 길어져 읽기 어려워졌고, 더 큰 문제가 있었다:
**커널을 새로 켤 때마다 정의 셀 15개를 순서대로 실행**해야 했고, 한 번은 예측을 저장하기 전에
커널이 초기화돼 **51회를 다시 학습**했다.

그래서 검증을 마친 정의를 전부 `src/pipeline.py`로 옮겼다.

| 파일 | 역할 |
|---|---|
| `src/metric.py` | 대회 공식 산식 (수정 금지) |
| `src/submission.py` | 제출 파일 생성·검증 |
| **`src/pipeline.py`** | **공통 정의 전부** — 데이터·fold·피처·풍속·가동률·라벨·LightGBM·채점 |
| `src/nn.py` | 신경망 — 산식손실 `MetricMLP` + 풍속 `WindMLP` |
| `notebooks/04_model_selection.ipynb` | **동결.** 1~21절 실험 원장(출력 포함). 실행하지 않는다 |
| **`notebooks/05_pipeline.ipynb`** | **여기.** 앞으로의 실험 |
| `train.ipynb` / `inference.ipynb` | 2차 평가용. **같은 `pipeline`을 import** (데드라인 2026-08-10) |

`04`는 지우지 않는다. 결론의 근거(오라클 실험·누수 수정·라벨 비교·T 스윕)가 거기 출력으로 남아 있다.

---

## 현재 위치 (2026-08-03)

| 제출 | 구성 | Total | 1-NMAE | FICR |
|---|---|---|---|---|
| v4 | LGB `B_norm` τ=0.50 | 0.6315 | 0.8620 | 0.4009 |
| v5 | + 산식손실 MLP **0.3** | 0.6413 | 0.8671 | **0.4155** |
| **v6** | + 산식손실 MLP **0.5** | **0.6416** ⭐ | 0.8687 | 0.4145 |
| v7 | + 산식손실 MLP 0.7 | 0.6405 | **0.8688** | 0.4123 |
| **1등** | | **0.67365** | 0.87964 | 0.46767 |

**격차 -0.0321. 그중 81%가 FICR이고, 그 FICR 격차의 98%가 "오차 10% 감소" 하나로 설명된다.**
1등은 산식 트릭을 쓴 게 아니라 더 정확하다. 그리고 우리 σ의 **89%는 예보 오차**다(04 노트북 14절).

## 확정된 최종 구성 (= v6)

```
1단계  풍속 2종(GBDT l2 결정적 · WindMLP) → 각각 LightGBM 발전량 예측 → 0.7 : 0.3
2단계  위 결과 0.5  +  산식손실 MetricMLP 0.5                        → 최종

  LightGBM 발전량 : quantile τ=0.50, 라벨 B_norm(가동률 정규화),
                    표본가중 actual/capacity(하한 0.1), 피처 상위 200, seed 5개
  산식손실 MLP    : 원본 라벨, 채점행만(이용률≥0.10), T_SOFT=0.006, full-batch,
                    256-256 GELU+BN+Dropout0.15, sigmoid 출력, 조기종료=대회산식, seed 5개
  풍속 GBDT       : l2, n_estimators=3000, 무작위성 없음(결정적), 입력 850개
```

## ⚠️ 검증 지표 규칙 — **주 지표는 `B안 평균`**

한때 "신경망 계열은 A안 우선"이라는 규칙을 세웠다가 **v6/v7 리더보드로 반증됐다.**

| MLP 블렌드 비중 | 0.3 | 0.5 | 0.7 | 0.8 |
|---|---|---|---|---|
| A안 | 0.6424 | 0.6429 | 0.6431 | **0.6438** ← 0.8이 최고라고 했다 |
| **B평균** | **0.6377** | **0.6377** | 0.6355 | 0.6344 |
| **리더보드(실측)** | 0.6413 | **0.6416** | 0.6405 | – |

**B평균이 맞았고 A안이 틀렸다.**
v5의 오프셋이 처음 양수(+0.0036)로 나오자 "비중을 더 올려야 한다"고 해석했는데,
그건 **모델 전체의 오프셋(절편)** 이지 **비중 곡선의 기울기**가 아니었다.

> **교훈: 오프셋이 바뀐 사실로부터 "어느 방향으로 더 가야 한다"를 유도하면 안 된다.**

**⇒ 모든 판정의 주 지표는 `B안 평균`. A안은 방향 일치 확인용 보조.** (CLAUDE.md 5장 원칙 그대로)

## 이 노트북의 구성

| 절 | 내용 | 학습 | 언제 |
|---|---|---|---|
| **1** | 셋업 + **이식 검증** (`pipeline.py`가 04를 정확히 옮겼나) | 36회 / 5분 | **가장 먼저** |
| **2** | ⭐⭐ **풍속 잔차 진단** — 1등까지의 길이 여기서 갈린다 | **0회** / 30분 | 본 작업 |
| **3** | 산식손실 MLP 튜닝 (λ → 학습량 → 피처 → 구조 → 정규화) | ~200회 / 40분 | 값싼 개선 |
| **4** | 최종 파이프라인 + 제출 파일 생성 | 51회 / 15분 | 구성이 바뀌었을 때만 |
| **5** | 로드맵 | – | |

**1절이 실패(`❌ 불일치`)하면 거기서 멈춘다.** `pipeline.py`를 신뢰할 수 없다는 뜻이다.
2절과 3절은 서로 독립이라 순서를 바꿔도 된다. **시간이 없으면 2절부터.**

---

# 1. 셋업 + 이식 검증

`src/pipeline.py`가 04의 정의를 **정확히** 옮겼는지 확인한다.
LightGBM은 결정적이므로 아래 두 값이 **소수점 넷째 자리까지** 재현돼야 한다.

| 구성 | A안(2024) | B안 평균 |
|---|---|---|
| `B_norm` τ=0.50 (v4~v6의 발전량 모델) | **0.6418** | **0.6356** |
| `A_asis` τ=0.60 (v2 구성) | **0.6391** | **0.6298** |

⏱️ 풍속 12회 + LightGBM 24회 = **36회, 약 5분.**
여기서 만든 캐시(`ctx.cache`, `PRED_CACHE`)를 2·3절이 그대로 재사용한다.

In [ ]:
import sys
import time
from pathlib import Path

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import numpy as np
import pandas as pd
import torch

import src.pipeline as pl
import src.nn as mnn                    # ⚠️ torch.nn과 충돌하므로 반드시 mnn
from src.metric import CAPACITY_KWH, TARGET_COLS

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

print("저장소 루트:", pl.REPO_ROOT)
ctx = pl.load_context(with_test=True, with_avail=True)
print("train:", ctx.train.shape, "| test:", ctx.test.shape)
print("공통 피처:", len(ctx.common_cols), "| 그룹 전용:", {g: len(v) for g, v in ctx.group_cols.items()})

print("\n=== 가동률 요약 (04의 20-1과 같아야 정상) ===")
rows = []
for g in pl.GROUP_COLS:
    a, lab = ctx.avail[g], ctx.train[g].notna()
    both = lab & a.notna()
    n_t = len(pl.TURBINES[g])
    rows.append({"그룹": g, "커버리지": round(both.sum() / lab.sum(), 4),
                 "평균 가동대수": round(a[both].mean() * n_t, 2),
                 "전대수 가동": round((a[both] >= 1 - 1e-9).mean(), 4)})
display(pd.DataFrame(rows).set_index("그룹"))
print("기대값: 커버리지 1.0/1.0/0.9647, 평균가동 5.81/5.84/4.81, 전대수 0.8435/0.8527/0.7719")

In [ ]:
# ── 이식 검증: 04의 두 기준선을 재현한다 (풍속 12 + LGB 24 = 36회) ──
PRED_CACHE = {}
stage_base = []

print("=== LGB_bnorm_q50 (v4~v6의 발전량 모델) ===")
stage_base.append(pl.run_variant(ctx, PRED_CACHE, "LGB_bnorm_q50",
                                 pl.lgbm_fold_fit_fn(mode="B_norm", tau=0.50)))
print("\n=== LGB_asis_q60 (v2 구성) ===")
stage_base.append(pl.run_variant(ctx, PRED_CACHE, "LGB_asis_q60",
                                 pl.lgbm_fold_fit_fn(mode="A_asis", tau=0.60)))

print("\n" + "=" * 70)
tbl = pl.summarize(stage_base)
display(tbl)

EXPECT = {"LGB_bnorm_q50": (0.6418, 0.6356), "LGB_asis_q60": (0.6391, 0.6298)}
ok = True
for name, (a_exp, b_exp) in EXPECT.items():
    a_got, b_got = tbl.loc[name, "A안(2024)"], tbl.loc[name, "B안 평균"]
    good = abs(a_got - a_exp) < 5e-4 and abs(b_got - b_exp) < 5e-4
    ok &= good
    print(f"  {name}: A안 {a_got:.4f}(기대 {a_exp}) / B평균 {b_got:.4f}(기대 {b_exp})  "
          f"{'✓' if good else '✗ 불일치!'}")
print("\n" + ("✅ 이식 검증 통과 — src/pipeline.py를 신뢰할 수 있다"
              if ok else "❌ 불일치. 여기서 멈추고 원인을 찾을 것 (05를 버리고 04로 돌아가야 할 수도)"))

**확인할 것**: **✅ 이식 검증 통과**가 떠야 합니다. `❌`면 멈추고 알려주세요.

---

# 2. ⭐⭐ 풍속 잔차 진단 — 1등까지의 길이 여기서 갈린다

## 2-0. 왜 이것이 최우선인가

04 노트북 14절 오라클 실험이 측정해둔 σ 분해:

```
σ_전체 0.166  =  풍속오차 0.146  ⊕  예측불가 0.079      (⊕ = 제곱합의 제곱근)
                    └─ 88% ─┘
```

"예측불가 0.079"는 **실측 나셀 풍속을 완벽히 알 때 남는 잔차**(`oracle_ws`)다.
FICR ≈ σ^-0.86 (오라클 실측 σ 0.079 / FICR 0.76으로 보정한 지수)으로 환산하면:

| 풍속 오차 | σ_전체 | 1-NMAE | FICR | **Total** |
|---|---|---|---|---|
| 현재 (v6) | 0.166 | 0.8687 | 0.4145 | **0.6416** |
| **-10%** | 0.153 | 0.8775 | 0.446 | **0.662** |
| **-20%** | 0.141 | 0.8871 | 0.478 | **0.683** |
| -30% | 0.129 | 0.8967 | 0.515 | 0.706 |
| (오라클) | 0.079 | – | 0.76 | 0.86 |
| **1등** | | 0.87964 | 0.46767 | **0.67365** |

**풍속 RMSE를 1.4~1.5 → 1.2~1.3 m/s로만 내리면 1등이다.**

반대로 풍속을 안 건드리는 개선은 전부 0.079 위에서 노는 것이라 상방이 막혀 있다.
실제로 20절(라벨 정제) +0.010, 21절(산식손실 MLP) +0.010, 블렌드 비중 +0.0003으로
**수확이 계속 줄어들고 있다.**

## 이 절이 답하는 질문

> **풍속 RMSE 1.5는 예보가 가진 정보의 한계인가, 우리 모델이 덜 짜낸 것인가?**

이걸 모르면 격자 CNN에 3일을 쓰고 아무것도 못 얻을 수 있다.
**현재 풍속 모델의 검증 구간 잔차(추정 − SCADA 실측)를 여러 축으로 쪼개**
조건부 편향이 남아 있는지 본다. 남아 있으면 = 아직 못 배운 구조가 있다.

## ⚖️ 사전 등록한 판정 규칙 (결과를 보고 기준을 바꾸지 않기 위해)

각 축에서 **`조건부 편향의 진폭 ÷ 잔차 σ`** 를 계산한다.

| 값 | 해석 | 행동 |
|---|---|---|
| **≥ 0.20** | 뚜렷한 **미학습 구조**가 있다 | 대응 피처를 만든다 |
| 0.10 ~ 0.20 | 약한 구조 | 비용이 낮으면(코드 몇 줄) 만든다 |
| < 0.10 | 사실상 없다 | 그 축은 포기 |

**모든 축이 0.10 미만이면** = 예보 정보의 한계에 가깝다.
평평한 피처를 아무리 더해도 소용없으므로 **격자 CNN(공간 구조)** 으로 직행한다.

⏱️ **학습 0회** (1절의 풍속 캐시 재사용). 분석은 즉시.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 2-1. 잔차 프레임 + 기본 통계
# ══════════════════════════════════════════════════════════════════
def wind_residual_frame(ctx):
    """검증 구간의 풍속 잔차를 fold×그룹으로 모아 하나의 긴 표로.
    잔차 = 우리 추정 − SCADA 실측 (양수 = 과대추정)."""
    parts = []
    for fold_name, info in ctx.fold_info.items():
        for g in pl.GROUP_COLS:
            vi = info["valid_idx"]
            est = pl.wind_estimate(ctx, g, info["cv_suffix"], info["train_mask"]).loc[vi]
            grid = pl.GROUP_NEAREST_LDAPS[g]
            d = pd.DataFrame({
                "fold": fold_name, "group": g,
                "kst_dtm": ctx.train.loc[vi, "kst_dtm"].to_numpy(),
                "est": est.to_numpy(dtype=float),
                "obs": ctx.train.loc[vi, f"scada_ws_{g}"].to_numpy(dtype=float),
                "fc10": ctx.train.loc[vi, f"{g}_ws10_nearest"].to_numpy(dtype=float),
                "wd": ctx.train.loc[vi, f"{g}_wd"].to_numpy(dtype=float),
                "blh": ctx.train.loc[vi, f"ldaps_g{grid}_etc_0_blh"].to_numpy(dtype=float),
            })
            d = d[d["obs"].notna()].copy()
            d["resid"] = d["est"] - d["obs"]                        # 양수 = 과대추정
            t = pd.to_datetime(d["kst_dtm"])
            d["hour"], d["month"] = t.dt.hour, t.dt.month
            d["wd_bin"] = (d["wd"] // 30 * 30).astype(int)          # 30도 구간
            d["ws_bin"] = (d["obs"] // 1.0)                         # 1 m/s 구간
            d["speedup"] = d["obs"] / d["fc10"].replace(0, np.nan)  # 실측 ÷ 10m 예보
            parts.append(d)
    return pd.concat(parts, ignore_index=True)


RES = wind_residual_frame(ctx)
SIGMA = RES.groupby("group")["resid"].std()
print(f"잔차 표: {len(RES):,}행 (fold 4 × 그룹 3, SCADA 결측 제외)")

print("\n=== 2-1a. 기본 성능 (fold × 그룹) ===")
display(RES.groupby(["fold", "group"]).apply(
    lambda d: pd.Series({"n": len(d), "RMSE": np.sqrt((d["resid"] ** 2).mean()),
                         "MAE": d["resid"].abs().mean(), "편향": d["resid"].mean(),
                         "σ": d["resid"].std(), "상관": d["est"].corr(d["obs"])}),
    include_groups=False).round(4))
print("16절 기준 RMSE: 1.410 / 1.530 / 1.521  ← 크게 다르면 이식 문제")

print("\n=== 2-1b. 그룹별 잔차 σ (모든 판정의 분모) ===")
print({g: round(SIGMA[g], 4) for g in pl.GROUP_COLS})

print("\n=== 2-1c. 평균 수축(regression to the mean) 점검 ===")
for g in pl.GROUP_COLS:
    d = RES[RES["group"] == g]
    print(f"  {g}: corr(잔차, 실측) = {d['resid'].corr(d['obs']):+.3f}   "
          f"corr(잔차, 추정) = {d['resid'].corr(d['est']):+.3f}")
print("""  ★ corr(잔차, 실측)이 강한 음수면 강풍을 과소·약풍을 과대 추정하고 있다는 뜻.
    발전량은 v³이라 **강풍 과소추정이 특히 비싸다.**""")

---

### 2-2. ⭐⭐ 풍향별 — 지형 speed-up이 있는가

**가덕산은 능선이다.** 능선 위에서는 기류가 압축돼 가속된다(Jackson–Hunt 이론).
그리고 **speed-up 계수는 풍향에 따라 크게 다르다** — 능선에 직각이면 크게, 나란하면 거의 없다.

우리는 수 km 해상도 예보 격자를 쓰는데 **그 해상도로는 능선 지형이 표현되지 않는다.**
따라서 `실측 ÷ 10m예보` 비율이 **풍향에 따라 체계적으로 출렁일 것**이다.

두 가지를 **따로** 본다.
1. **`speedup`의 풍향별 곡선** — 지형 효과가 존재하는가
2. **잔차의 풍향별 평균** — 우리 모델이 그걸 **이미 학습했는가**

⚠️ **1번이 출렁이는데 2번이 평평하면 모델이 이미 잡은 것**이라 새 피처가 필요 없다.
**둘 다 출렁여야** 피처화할 값어치가 있다.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 2-2. 풍향별 — 지형 speed-up + 잔차
# ══════════════════════════════════════════════════════════════════
sp = RES[RES["fc10"] > 2.0]        # 약풍은 비율이 불안정하므로 제외

print("=== 2-2a. speedup(실측 ÷ 10m예보) 풍향 30도 구간별 중앙값 ===")
piv_sp = sp.pivot_table(index="wd_bin", columns="group", values="speedup", aggfunc="median").round(3)
piv_n = sp.pivot_table(index="wd_bin", columns="group", values="speedup", aggfunc="size")
display(pd.concat([piv_sp, piv_n.add_suffix("_n")], axis=1))
print("진폭(최대-최소):", {g: round(piv_sp[g].max() - piv_sp[g].min(), 3) for g in pl.GROUP_COLS})
print("★ 표본수(_n) 200 미만 구간의 값은 믿지 말 것")

print("\n=== 2-2b. 우리 모델의 잔차 — 풍향별 평균 (이미 학습했나) ===")
piv_r = RES.pivot_table(index="wd_bin", columns="group", values="resid", aggfunc="mean").round(3)
display(piv_r)


def verdict(piv, label):
    """사전 등록 규칙: 조건부 편향 진폭 ÷ 잔차 σ."""
    rows = []
    for g in pl.GROUP_COLS:
        amp = float(piv[g].max() - piv[g].min())
        r = amp / SIGMA[g]
        rows.append({"축": label, "group": g, "편향 진폭": round(amp, 3),
                     "잔차 σ": round(SIGMA[g], 3), "진폭÷σ": round(r, 3),
                     "판정": "★피처화" if r >= 0.20 else ("△저비용이면" if r >= 0.10 else "무시")})
    return pd.DataFrame(rows)


VERDICTS = [verdict(piv_r, "풍향")]
print("\n=== 2-2c. 판정 ===")
display(VERDICTS[-1].set_index(["축", "group"]))

---

### 2-3. 시간대 · 경계층높이 · 월 — 대기 안정도가 빠져 있는가

10m 예보에서 117m를 추정하는 문제의 **물리적 핵심**이다.
멱법칙 `v(z) = v(10)·(z/10)^α`에서 지수 α는 대기 안정도가 결정한다.

- **안정(야간 복사냉각·역전층)**: α ≈ 0.3~0.4 → 117m가 10m의 **1.9배**
- **불안정(주간 대류)**: α ≈ 0.1 → **1.3배**

**같은 10m 풍속인데 허브고도가 45% 차이난다.**
안정도 지표가 피처에 없으므로(있는 건 `blh`뿐) 야간/주간에 체계적 편향이 남아 있을 수 있다.

`blh`(경계층 높이)는 안정도의 대리지표다 — 낮으면 안정, 높으면 불안정.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 2-3. 시간대 · blh · 월
# ══════════════════════════════════════════════════════════════════
print("=== 2-3a. 시간대별 잔차 평균 ===")
piv_h = RES.pivot_table(index="hour", columns="group", values="resid", aggfunc="mean").round(3)
display(piv_h.T)

night = RES[RES["hour"].between(21, 23) | RES["hour"].between(0, 5)]
day = RES[RES["hour"].between(10, 16)]
print("야간(21~05시) 편향:", {g: round(night[night['group'] == g]['resid'].mean(), 3) for g in pl.GROUP_COLS})
print("주간(10~16시) 편향:", {g: round(day[day['group'] == g]['resid'].mean(), 3) for g in pl.GROUP_COLS})

print("\n=== 2-3b. speedup 야간 vs 주간 — 시어의 물리 확인 ===")
for g in pl.GROUP_COLS:
    n_ = night[(night["group"] == g) & (night["fc10"] > 2)]["speedup"].median()
    d_ = day[(day["group"] == g) & (day["fc10"] > 2)]["speedup"].median()
    print(f"  {g}: 야간 {n_:.3f} / 주간 {d_:.3f}   차이 {n_ - d_:+.3f}")
print("  ★ 야간이 뚜렷하게 크면(시어가 큼) 대기 안정도가 실재한다는 물리적 증거다.")

print("\n=== 2-3c. 경계층 높이(blh) 사분위별 잔차 ===")
RES["blh_q"] = RES.groupby("group")["blh"].transform(
    lambda s: pd.qcut(s, 4, labels=["Q1(낮음/안정)", "Q2", "Q3", "Q4(높음/불안정)"]))
piv_b = RES.pivot_table(index="blh_q", columns="group", values="resid", aggfunc="mean", observed=True).round(3)
display(piv_b)

print("\n=== 2-3d. 월별 잔차 ===")
piv_m = RES.pivot_table(index="month", columns="group", values="resid", aggfunc="mean").round(3)
display(piv_m.T)

VERDICTS += [verdict(piv_h, "시간대"), verdict(piv_b, "blh 사분위"), verdict(piv_m, "월")]
print("\n=== 2-3e. 판정 ===")
display(pd.concat(VERDICTS[1:]).set_index(["축", "group"]))

---

### 2-4. 풍속 구간 · 자기상관 · 그룹 간 상관

- **풍속 구간별**: 강풍에서 편향이 크면 외삽 문제. 발전량은 v³이라 **ramp 구간(3~12 m/s) 편향이 가장 비싸다**
- **시간 자기상관**: 크면 **위상 오차**(전선 통과 시각이 어긋남)
  ⚠️ 예측 시점에 과거 실측을 못 쓰므로 사후 보정에는 못 쓴다. **예보 자체의 시간 구조로만** 잡을 수 있다
- **그룹 간 상관**: 높으면 세 그룹이 **같은 이유로** 틀린다 = 단지 전체에 걸친 예보 오차.
  그룹별 미세조정보다 **예보를 더 잘 읽는 것(격자 CNN)** 이 답이라는 신호

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 2-4. 풍속 구간 · 자기상관 · 그룹 간 상관
# ══════════════════════════════════════════════════════════════════
print("=== 2-4a. 실측 풍속 구간(1 m/s)별 잔차 평균 ===")
sub = RES[RES["ws_bin"].between(1, 19)]
piv_w = sub.pivot_table(index="ws_bin", columns="group", values="resid", aggfunc="mean").round(3)
cnt_w = sub.pivot_table(index="ws_bin", columns="group", values="resid", aggfunc="size")
display(pd.concat([piv_w, cnt_w.add_suffix("_n")], axis=1))
VERDICTS.append(verdict(piv_w[piv_w.index.isin(cnt_w[cnt_w.min(axis=1) >= 100].index)], "풍속 구간"))

print("\n=== 2-4b. 잔차의 시간 자기상관 (A안 fold, 위상 오차) ===")
a = RES[RES["fold"] == "A안(2024)"].sort_values("kst_dtm")
for g in pl.GROUP_COLS:
    s = a[a["group"] == g].set_index("kst_dtm")["resid"]
    print(f"  {g}: lag 1/2/3/6/12/24h = {[round(s.autocorr(lag=k), 3) for k in (1, 2, 3, 6, 12, 24)]}")
print("  ★ lag1 > 0.5면 잔차가 강하게 이어진다 = 구조적 원인(위상 오차 등)이 있다는 뜻")

print("\n=== 2-4c. 그룹 간 잔차 상관 (공통 오차 = 격자 CNN이 노릴 대상) ===")
w = a.pivot_table(index="kst_dtm", columns="group", values="resid")
display(w.corr().round(3))
print(f"  평균 상관 ≈ {w.corr().values[np.triu_indices(3, 1)].mean():.2f}")
print("  ★ 0.7 이상이면 세 그룹이 같은 이유로 틀린다 = 예보 자체를 더 잘 읽어야 한다")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 2-5. 종합 판정 — 무엇을 만들 것인가
# ══════════════════════════════════════════════════════════════════
SUM = pd.concat(VERDICTS, ignore_index=True)
print("=== 축별 최대 진폭÷σ (그룹 중 최댓값 기준) ===")
best = SUM.groupby("축")["진폭÷σ"].max().sort_values(ascending=False)
CARDS = {
    "풍향": "지형 speed-up 배율 (풍향별 실측÷예보 중앙값을 학습구간에서 적합 → 연속 피처)",
    "시간대": "대기 안정도 (벌크 리처드슨 수 근사 또는 blh × 야간여부)",
    "blh 사분위": "대기 안정도 (동상)",
    "월": "계절 피처는 이미 있음(month_sin/cos) — 크면 다른 원인 의심",
    "풍속 구간": "파워커브 기울기 가중 손실 (sample_weight ∝ |dP/dv|)",
}
rows = []
for axis, v in best.items():
    rows.append({"축": axis, "최대 진폭÷σ": round(v, 3),
                 "판정": "★피처화" if v >= 0.20 else ("△저비용이면" if v >= 0.10 else "무시"),
                 "대응 카드": CARDS.get(axis, "")})
display(pd.DataFrame(rows).set_index("축"))

print(f"""
=== 최종 결론 ===
  · ★피처화 축이 있으면 → 그 카드부터 만든다 (각 12회 학습)
  · 전부 '무시'(0.10 미만)이면 → **예보 정보의 한계에 가깝다.**
    평평한 피처를 더 만들어도 소용없으므로 **격자 CNN(공간 구조)** 으로 직행한다

⚠️ 04에서 배운 것을 여기서도 지킬 것
  · **구간 더미를 만들지 말 것.** regime_calm/ramp/rated·high_wind_caution이 전부 gain 0.000%였다.
    도메인 지식은 '어떤 변수를 만들지'에 쓰고 '어떻게 이산화할지'는 모델에 맡긴다
  · speed-up 배율은 **학습 구간에서만 적합**해야 한다 (fold-safe). 검증 구간 실측을 보면 누수다
""")

---

# 3. 산식손실 MLP 튜닝 — 여태 `T_SOFT`만 건드렸다

## 3-0. 정직한 재고 조사

| 대상 | 상태 |
|---|---|
| 풍속 LightGBM | ✅ 5방향 시도 → **전부 기본값을 못 이김** (04의 18-2/18-3). 유용한 negative result |
| 발전량 LightGBM | ⚠️ 손실·τ·표본가중·피처수만. **`learning_rate`·`num_leaves`·정규화는 기본값** |
| **산식손실 MLP** | ❌ **`T_SOFT`만 스윕(21-3).** 나머지는 **이전 프로젝트가 179피처로 정한 값** |
| 풍속 MLP | ❌ 없음 (512-256-128, 임의 결정) |
| 블렌드 비중 | ✅ **0.5 확정** (v5~v7 리더보드) |

**왜 MLP부터인가**
1. v5에서 MLP가 리더보드 **+0.0098**을 만들었다. 주력 모델이다
2. **값이 남의 것이다.** 179피처·다른 풍속·다른 라벨 처리에서 정해진 값이 우리에게 최적일 이유가 없다
3. **비용이 싸다.** MLP 1회 10초 → 한 설정을 전 fold 검증하는 데 12회 = **약 2분**

## 탐색 방식 — 좌표별(one-factor-at-a-time), 격자 아님

전체 격자는 설정이 수백 개가 되고 **다중비교로 가짜 승자**가 나온다.
근거가 강한 순서대로 **한 번에 하나씩** 바꾸고, 이긴 것만 다음 단계로 물려준다.

| 소절 | 바꾸는 것 | 근거 |
|---|---|---|
| **3-1** | **FICR 항 가중치 λ** | **v5~v7 리더보드가 만들어준 가설.** 근거가 가장 신선하다 |
| 3-2 | **학습량** (`patience`, `max_epochs`) | full-batch라 1에폭 = 기울기 1스텝인데 61에폭에 멈췄다 |
| 3-3 | **피처 세트** (200 / 400 / 전체) | 트리 중요도로 고른 200개가 매끄러운 모델에 맞을 이유가 없다 |
| 3-4 | **구조** (폭·깊이) | 179피처에 맞춰 정한 값 |
| 3-5 | **정규화** (dropout, weight decay) | 구조를 바꿨으면 함께 조정 |
| 3-6 | 최종 후보 **시드 5개 + 블렌드 곡선** | 다중비교 방어 |

## ⚠️ 판정 규칙 — **자동으로 적용된다**
- **주 지표는 `B안 평균`** (A안 우선 규칙은 폐기됨)
- 시드 간 σ가 **0.0024**였으므로 문턱은 그 2배인 **+0.005**.
  각 소절 끝의 `pick()`이 **기준선 대비 +0.005 이상인 후보만 자동 채택**하고,
  아니면 기준 설정을 그대로 다음 소절로 물려준다 (다중비교 방어)
- 표를 보고 직접 정하고 싶으면 각 소절의 주석 처리된 줄을 풀어 덮어쓰면 된다
- 3-1~3-5에서 **아무것도 문턱을 못 넘으면 3-6이 자동으로 축소 실행**된다(60회 절약)

**⇒ 3절은 위에서 아래로 그냥 실행해도 된다.**

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 3-0. 튜닝 공용 도구
# ══════════════════════════════════════════════════════════════════
MLP_CACHE = dict(PRED_CACHE)      # 1절의 LightGBM 예측을 함께 담아 블렌드까지 볼 수 있게
stage_mlp = []


def try_cfg(label, cfg, seeds=(pl.SEED,)):
    """설정 하나를 전 fold에서 돌리고 점수표를 돌려준다."""
    t0 = time.time()
    print(f"=== {label} ===  {({k: v for k, v in cfg.items() if pl.MLP_CFG.get(k) != v}) or '기준선'}")
    d = pl.run_variant(ctx, MLP_CACHE, label, pl.metric_mlp_fit_fn(cfg, seeds=seeds))
    stage_mlp.append(d)
    print(f"    ({time.time() - t0:.0f}초)")
    return d


def show(sort="B안 평균"):
    return pl.summarize(stage_mlp).sort_values(sort, ascending=False)


def decompose(labels):
    """1-NMAE / FICR 분해 — 어느 축에서 얻고 잃었는지."""
    rows = []
    for lb in labels:
        d = pd.concat([x for x in stage_mlp if x["model"].iloc[0] == lb])
        b = d[d["fold"].isin(pl.B_FOLDS)]
        rows.append({"변형": lb, "1-NMAE(B)": round(b["1-NMAE"].mean(), 4),
                     "FICR(B)": round(b["FICR"].mean(), 4),
                     "Score(B)": round(b["score"].mean(), 4),
                     "A안": round(d[d["fold"] == "A안(2024)"]["score"].iloc[0], 4)})
    return pd.DataFrame(rows).set_index("변형")


THRESHOLD = 0.005      # 시드 간 σ가 0.0024였으므로 그 2배를 문턱으로 (사전 등록)


def pick(cands, base_label):
    """사전 등록 규칙대로 승자를 **자동 선택**한다.

    cands : {라벨: 설정}. base_label은 그중 '현행 설정'에 해당하는 라벨.
    B안 평균이 기준선보다 **+THRESHOLD 이상** 좋은 후보만 채택하고,
    아니면 기준선을 그대로 물려준다. (다중비교로 인한 가짜 승자 방어)
    """
    t = pl.summarize(stage_mlp)
    b0 = t.loc[base_label, "B안 평균"]
    best_lb = max((lb for lb in cands if lb in t.index), key=lambda lb: t.loc[lb, "B안 평균"])
    best_b = t.loc[best_lb, "B안 평균"]
    adopt = (best_lb != base_label) and (best_b - b0 >= THRESHOLD)
    print(f"\n  ▶ 기준 {base_label} B평균 {b0:.4f} | 최고 {best_lb} {best_b:.4f} "
          f"(차이 {best_b - b0:+.4f}, 문턱 +{THRESHOLD})")
    print(f"    → {'✅ 채택: ' + best_lb if adopt else '❌ 문턱 미달 — 기준 설정 유지'}")
    print("    ※ 표를 보고 직접 정하려면 아래 줄에서 반환값을 덮어쓰세요")
    return dict(cands[best_lb] if adopt else cands[base_label])


BASE = dict(pl.MLP_CFG)
print("기준 설정:", BASE)
print(f"\n채택 문턱: B안 평균 +{THRESHOLD} (시드 간 σ 0.0024의 2배)")
print("참고 — LightGBM 단독(v4~v6 발전량 모델): A안 0.6418 / B평균 0.6356")
print("       21절 MLP(기준선) 단독: A안 0.6432 / B평균 0.6307")

---

### 3-1. ⭐ FICR 항 가중치 λ — v5~v7 리더보드가 만들어준 가설

#### 관찰
MLP 블렌드 비중을 올렸더니:

| 비중 | 0.3 (v5) | 0.5 (v6) | 0.7 (v7) |
|---|---|---|---|
| Total | 0.6413 | **0.6416** | 0.6405 |
| 1-NMAE | 0.8671 | 0.8687 | **0.8688** (오르다 포화) |
| FICR | **0.4155** | 0.4145 | 0.4123 (단조 하락) |

**대회 산식을 손실로 쓴 모델인데 정작 FICR은 LightGBM보다 나쁘다**(로컬도 MLP 0.3877 vs LGB 0.4036).
MLP는 **NMAE 쪽으로 치우쳐** 있다.

#### 가설
점수는 `0.5·(1-NMAE) + 0.5·FICR`로 두 항이 대등하지만 **남은 여지는 전혀 대등하지 않다.**
1-NMAE는 **0.869로 이미 1에 가깝고** FICR은 **0.414로 멀다.**

```
L = 0.5·nmae − 0.5·λ·ficr_soft        # λ ∈ {1(현행), 1.5, 2, 3}
```

λ>1은 **"NMAE를 조금 내주고 밴드 안으로 들어가라"** 는 지시다.

⚠️ **λ≠1은 대회 산식이 아닌 다른 목적함수**를 최적화한다.
반드시 **실제 점수(계단 그대로)** 로 검증해 이득이 있을 때만 쓴다.

#### 미리 적어두는 실패 양상
λ가 과하면 모델이 **밴드 안에 못 들어갈 표본을 아예 포기**하고 극단적으로 예측할 수 있다.
그러면 FICR은 조금 오르고 **1-NMAE가 크게 무너진다.** 두 축을 같이 봐야 하는 이유다.

⏱️ 4설정 × 12 = **48회, 약 8분.**

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 3-1. FICR 항 가중치 λ
# ══════════════════════════════════════════════════════════════════
LAMS = [1.0, 1.5, 2.0, 3.0]
cands1 = {f"L_lam{lam}": {**BASE, "ficr_weight": lam} for lam in LAMS}
for lb, cfg in cands1.items():
    try_cfg(lb, cfg)

print("\n" + "=" * 70)
display(show())
print("\n=== λ별 1-NMAE / FICR 분해 (두 축을 같이 볼 것) ===")
display(decompose(list(cands1)))
print("""
★ 읽는 법
  · FICR이 오르면서 Score(B평균)도 오르면 가설 지지
  · FICR만 오르고 1-NMAE가 그 이상 무너지면 → '남은 여지' 가설 기각
  · λ가 클수록 단조 개선이면 4.0/5.0도 추가로 볼 것""")

BEST1 = pick(cands1, "L_lam1.0")
# BEST1 = {**BASE, "ficr_weight": 2.0}      # ← 표를 보고 직접 정하려면 이 줄의 주석을 푸세요
print("\n다음 소절로 물려줄 설정:", {k: v for k, v in BEST1.items() if pl.MLP_CFG.get(k) != v} or "기준선")

---

### 3-2. 학습량 — 기울기 스텝이 61번밖에 안 됐다

**full-batch라 1에폭 = 기울기 1스텝**이다. 그런데 21-2 시험 학습이 **61에폭**에서 최적을 찍었다.
즉 **파라미터 12만 개짜리 모델이 기울기 갱신 61번으로 학습을 끝냈다.**
`PATIENCE=60`이 너무 빨리 잘랐을 가능성이 크다.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 3-2. 학습량
# ══════════════════════════════════════════════════════════════════
cands2 = {"E_base": dict(BEST1),
          "E_pat120": {**BEST1, "patience": 120},
          "E_pat120_ep800": {**BEST1, "patience": 120, "max_epochs": 800}}
for lb, cfg in cands2.items():
    try_cfg(lb, cfg)

# 실제로 몇 에폭을 썼는지 (조기 종료가 언제 걸렸나)
print("\n=== 설정별 사용 에폭 (A안 fold) ===")
info = ctx.fold_info["A안(2024)"]
for lb in ["E_base", "E_pat120"]:
    cfg = cands2[lb]
    eps = {g.split("_")[-1]: pl.metric_mlp_predict(ctx, g, info["cv_suffix"], info["train_mask"], cfg)[1]
           for g in pl.GROUP_COLS}
    print(f"  {lb}: {eps}  (max_epochs={cfg['max_epochs']})")
print("★ max_epochs에 한참 못 미치면 patience가 일찍 자른 것이다.")

display(show())
display(decompose(list(cands2)))
BEST2 = pick(cands2, "E_base")

---

### 3-3. 피처 세트 — 트리가 고른 200개가 MLP에도 맞을까

지금 MLP는 **LightGBM 중요도 상위 200개**를 쓴다. 그런데 둘은 피처를 쓰는 방식이 다르다.

- **트리**: 한 번에 한 변수로 자른다 → 상관 높은 피처 중 **하나만** 있으면 된다
- **신경망**: 모든 입력의 선형결합을 만든다 → **약한 신호가 많이 모이면 도움이 된다.**
  트리에게 gain 0이던 피처가 신경망에는 쓸모 있을 수 있다

04의 15-2에서 "LightGBM은 top50까지 줄여도 점수가 같다"고 확인했는데 **그건 트리 얘기다.**

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 3-3. 피처 세트
# ══════════════════════════════════════════════════════════════════
cands3 = {"F_top200": {**BEST2, "top_n": 200},
          "F_top400": {**BEST2, "top_n": 400},
          "F_all": {**BEST2, "top_n": "all"}}
for lb, cfg in cands3.items():
    try_cfg(lb, cfg)

display(show())
display(decompose(list(cands3)))
BEST3 = pick(cands3, "F_top200")
print("""
★ 피처를 늘려 좋아지면 → 트리 중요도 선택이 MLP에 최적이 아니었다는 뜻.
  그러면 LightGBM과 MLP가 **서로 다른 피처 집합**을 쓰게 되어 블렌드 다양성도 올라간다.
★ 나빠지면 → 표본 1.5만에 피처 850개는 과적합. 200개가 맞았다.""")

---

### 3-4. 구조 — 폭과 깊이

현재 `256 → 256`. 이전 프로젝트가 **179피처**에 맞춰 정한 값이다.
피처가 200~850개로 늘면 첫 층이 병목이 될 수 있다.

**축소 후보(128×2)를 일부러 넣는다.** 확대만 시험하면 "더 큰 게 낫다"는 결론밖에 안 나온다.
**양쪽을 봐야 어느 방향이 맞는지 안다.**

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 3-4. 구조
# ══════════════════════════════════════════════════════════════════
cands4 = {f"H_{'_'.join(map(str, hid))}": {**BEST3, "hidden": hid}
          for hid in [(256, 256), (512, 512), (512, 256, 128), (128, 128)]}
for lb, cfg in cands4.items():
    try_cfg(lb, cfg)

display(show())
BEST4 = pick(cands4, "H_256_256")
print("★ 문턱을 못 넘으면 자동으로 현행(256×2)이 유지된다 — 빠르고 과적합 위험이 낮은 쪽이다.")

---

### 3-5. 정규화

구조를 바꿨다면 정규화도 함께 봐야 한다. 현재 `dropout 0.15`, `weight_decay 1e-4`.

⚠️ 여기까지 오면 비교한 설정이 **17개**다. 다중비교 편향이 쌓인다.
**3-6에서 시드 5개로 재확인**하고, 거기서도 이겨야 채택한다.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 3-5. 정규화
# ══════════════════════════════════════════════════════════════════
cands5 = {"R_base": dict(BEST4),
          "R_d025": {**BEST4, "p_drop": 0.25},
          "R_d005": {**BEST4, "p_drop": 0.05},
          "R_wd1e3": {**BEST4, "weight_decay": 1e-3}}
for lb, cfg in cands5.items():
    try_cfg(lb, cfg)

display(show())
BEST_CFG = pick(cands5, "R_base")

print("\n" + "=" * 70)
CHANGED = {k: v for k, v in BEST_CFG.items() if pl.MLP_CFG.get(k) != v}
print("★ 3절 최종 결과:", CHANGED or "**바뀐 것 없음 — 현행 설정이 이미 최적**")

---

### 3-6. 최종 확인 — 시드 5개 + 블렌드 곡선

앞에서 고른 설정을 **시드 5개**로 다시 돌리고, LightGBM과 블렌드했을 때까지 확인한다.
**여기서 기준선(21절 설정 = 리더보드 0.6416을 만든 그 MLP)을 못 이기면 채택하지 않는다.**

⏱️ 2설정 × 5시드 × 12 = 120회, 약 20분.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 3-6. 최종 확인
# ══════════════════════════════════════════════════════════════════
MLP_SEEDS = [pl.SEED, 7, 123, 2024, 31]
print("최종 후보:", CHANGED or "기준선과 동일")

try_cfg("MLP_old_s5", BASE, seeds=MLP_SEEDS)        # 21절 설정 = v6의 MLP
VARS = ["MLP_old_s5"]
if CHANGED:
    try_cfg("MLP_new_s5", BEST_CFG, seeds=MLP_SEEDS)    # 튜닝 결과
    VARS.append("MLP_new_s5")
else:
    print("\n⏭️ 3-1~3-5에서 문턱을 넘은 설정이 없어 MLP_new는 건너뜁니다(60회 절약).")
    print("   → 현행 MLP가 이미 좋다는 뜻. 2절(풍속)로 넘어가세요.")

print("\n=== LightGBM + MLP 블렌드 곡선 (주 지표: B안 평균) ===")
rows = []
for var in VARS:
    for w in np.arange(0.0, 1.01, 0.1):
        a_, b_, mn_, _ = pl.blend_score(ctx, MLP_CACHE, "LGB_bnorm_q50", var, w)
        rows.append({"MLP": var, "비중": round(w, 1), "A안": round(a_, 4),
                     "B평균": round(b_, 4), "B최솟값": round(mn_, 4)})
curve = pd.DataFrame(rows)
print("[B안 평균]")
display(curve.pivot(index="비중", columns="MLP", values="B평균").round(4))
print("[B안 최솟값]")
display(curve.pivot(index="비중", columns="MLP", values="B최솟값").round(4))
print("""
★ 채택 조건 (전부 만족해야)
  1. MLP_new 단독이 MLP_old 단독보다 B평균에서 +0.005 이상 높다
  2. 블렌드 최고점도 MLP_new 쪽이 높다
  3. B안 최솟값이 나빠지지 않았다
  ⇒ 만족하면 src/pipeline.py의 MLP_CFG를 고치고, models/v5_parts.npz를 지운 뒤 4절을 다시 실행

  참고 — 현행(v6) 로컬값: LGB 단독 B평균 0.6356 / 블렌드 w=0.5 B평균 0.6377""")

---

# 4. 최종 파이프라인 + 제출 파일

**구성이 바뀌었을 때만 실행한다.** 지금은 v6가 이미 제출돼 있고 `models/v5_parts.npz`에
예측 부분이 저장돼 있으므로, **블렌드 비중만 바꾸는 실험은 학습 0회**로 가능하다.

| 상황 | 할 일 |
|---|---|
| 블렌드 비중만 바꾼다 | 4-2만 실행 (**학습 0회**) |
| MLP 설정·피처·라벨이 바뀌었다 | **`models/v5_parts.npz`를 지우고** 4-1부터 (51회, 15분) |

⏱️ 4-1: 풍속 6 + LightGBM 30 + 산식손실 MLP 15 = **51회, 약 15분**

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 4-1. test 예측의 두 부분 만들기 (저장 파일이 있으면 건너뜀)
# ══════════════════════════════════════════════════════════════════
import src.submission as subm
from src.submission import build_submission, validate_submission, save_submission

SAMPLE_PATH = pl.REPO_ROOT / "data" / "sample_submission.csv"
subm.SAMPLE_SUBMISSION_PATH = SAMPLE_PATH
subm.SUBMISSIONS_DIR = pl.REPO_ROOT / "submissions"    # ⚠️ 둘 다 덮어써야 한다 (04의 17-4 함정)
PARTS_PATH = pl.REPO_ROOT / "models" / "v5_parts.npz"
PARTS_PATH.parent.mkdir(parents=True, exist_ok=True)

FINAL_SEEDS = [pl.SEED, 7, 123, 2024, 31]
WIND_BLEND = {"gbdt": 0.7, "mlp": 0.3}      # 1단계: 풍속 2종 (04의 18-5)
TAU, LABEL_MODE = pl.DEFAULT_TAU, pl.DEFAULT_LABEL_MODE
MLP_CFG_FINAL = dict(pl.MLP_CFG)            # 3절에서 채택한 게 있으면 반영하세요

_g = ctx.test["kst_dtm"].diff().dropna().unique()
assert ctx.test["kst_dtm"].is_monotonic_increasing and len(_g) == 1 and _g[0] == pd.Timedelta("1h")
print("✓ test 시간축 1시간 연속")


def build_final_parts():
    """전체 train으로 학습해 test 예측의 두 부분(lgb_part, mlp_part)을 만든다."""
    full = ctx.full_mask
    ws = {}

    print("\n=== 풍속 GBDT (결정적) ===")
    for g in pl.GROUP_COLS:
        m = pl.fit_wind_gbdt(ctx, g, full, "l2")
        Xtr, Xte = pl.build_wind_input_frame(ctx, ctx.train, g), pl.build_wind_input_frame(ctx, ctx.test, g)
        assert list(Xtr.columns) == list(Xte.columns), f"{g}: 풍속 입력 컬럼 불일치"
        ws[("gbdt", g)] = (pd.Series(m.predict(Xtr), index=ctx.train.index).clip(lower=0.0),
                           pd.Series(m.predict(Xte), index=ctx.test.index).clip(lower=0.0))
        print(f"  {g}: train 내 상관 {ws[('gbdt', g)][0].corr(ctx.train[f'scada_ws_{g}']):.4f}")

    print("\n=== 풍속 MLP ===")
    for g in pl.GROUP_COLS:
        Xtr, Xte = pl.build_wind_input_frame(ctx, ctx.train, g), pl.build_wind_input_frame(ctx, ctx.test, g)
        y = ctx.train[f"scada_ws_{g}"]
        fi = y.notna()
        cut = ctx.train.loc[fi, "kst_dtm"].quantile(0.9)
        o_tr, o_te = mnn.fit_wind_mlp(Xtr, y, fi & (ctx.train["kst_dtm"] <= cut),
                                      fi & (ctx.train["kst_dtm"] > cut), [Xtr, Xte], seed=pl.SEED)
        ws[("mlp", g)] = (pd.Series(o_tr, index=ctx.train.index), pd.Series(o_te, index=ctx.test.index))
        print(f"  {g}: train 내 상관 {ws[('mlp', g)][0].corr(y):.4f}")

    print("\n=== LightGBM 발전량 (풍속 2종 × seed 5) ===")
    lgb_pred, keep_pc = {}, {}
    for src_ in ["gbdt", "mlp"]:
        for g in pl.GROUP_COLS:
            ws_tr, ws_te = ws[(src_, g)]
            ok = ctx.train[g].notna()
            edges, vals = pl.fit_power_curve_oracle(ws_tr[ok].to_numpy(float),
                                                    ctx.train.loc[ok, g].to_numpy(float))
            tag = f"fin_{src_}"
            Xtr = pl.build_frame_given_pc(ctx, ctx.train, g, ws_tr, edges, vals, tag)
            Xte = pl.build_frame_given_pc(ctx, ctx.test, g, ws_te, edges, vals, tag)
            assert list(Xtr.columns) == list(Xte.columns), f"{src_}/{g}: 입력 컬럼 불일치"

            m0 = pl.lgbm_train_label(ctx, Xtr, g, full, TAU, pl.SEED, LABEL_MODE)
            keep = pl.select_features(m0, Xtr.columns, pl.BEST_TOPN)
            preds = [pl.lgbm_train_label(ctx, Xtr[keep], g, full, TAU, sd, LABEL_MODE, rand=True)
                     .predict(Xte[keep]) for sd in FINAL_SEEDS]
            lgb_pred[(src_, g)] = np.clip(np.mean(preds, axis=0), 0, CAPACITY_KWH[g])
            if src_ == "gbdt":
                keep_pc[g] = (edges, vals, keep)
            print(f"  ✓ {src_}/{g}: 이용률 {lgb_pred[(src_, g)].mean() / CAPACITY_KWH[g] * 100:.1f}%")

    lgb_part = {g: sum(WIND_BLEND[s] * lgb_pred[(s, g)] for s in WIND_BLEND) for g in pl.GROUP_COLS}

    print("\n=== 산식손실 MLP (GBDT 풍속, 시드 5) ===")
    mlp_part = {}
    for g in pl.GROUP_COLS:
        cap = CAPACITY_KWH[g]
        ws_tr, ws_te = ws[("gbdt", g)]
        edges, vals, keep = keep_pc[g]
        Xtr = pl.build_frame_given_pc(ctx, ctx.train, g, ws_tr, edges, vals, "fin_gbdt")[keep].to_numpy(float)
        Xte = pl.build_frame_given_pc(ctx, ctx.test, g, ws_te, edges, vals, "fin_gbdt")[keep].to_numpy(float)

        ratio = (ctx.train[g] / cap).to_numpy(dtype=float)
        fi = ctx.train[g].notna().to_numpy() & (ratio >= mnn.EVAL_MIN_RATIO)   # 채점 대상만
        times = ctx.train["kst_dtm"].to_numpy()
        cut = pd.Series(ctx.train.loc[fi, "kst_dtm"]).quantile(0.9)
        tr, es = fi & (times <= np.datetime64(cut)), fi & (times > np.datetime64(cut))

        mu, sd_ = mnn.fit_standardizer(Xtr[tr])          # 표준화는 학습 구간에서만
        Xes_t = torch.tensor(((Xtr[es] - mu) / sd_).astype(np.float32))
        y_es = ratio[es]

        def _eval(model, _X=Xes_t, _y=y_es):
            with torch.no_grad():
                return mnn.group_score(_y, np.clip(model(_X).numpy(), 0.0, 1.0))

        sp = []
        for s_ in FINAL_SEEDS:
            model, ep = mnn.train_metric_mlp(
                (Xtr[tr] - mu) / sd_, ratio[tr], seed=s_, n_epochs=MLP_CFG_FINAL["max_epochs"],
                t_soft=MLP_CFG_FINAL["t_soft"], eval_fn=_eval, hidden=MLP_CFG_FINAL["hidden"],
                p_drop=MLP_CFG_FINAL["p_drop"], lr=MLP_CFG_FINAL["lr"],
                weight_decay=MLP_CFG_FINAL["weight_decay"], patience=MLP_CFG_FINAL["patience"],
                eval_every=MLP_CFG_FINAL["eval_every"], ficr_weight=MLP_CFG_FINAL["ficr_weight"])
            sp.append(mnn.predict_ratio(model, Xte, mu, sd_))
            print(f"    {g}/seed{s_}: {ep}에폭")
        mlp_part[g] = np.mean(sp, axis=0) * cap
        print(f"  ✓ {g}: 이용률 {mlp_part[g].mean() / cap * 100:.1f}%")
    return lgb_part, mlp_part


if PARTS_PATH.exists():
    _z = np.load(PARTS_PATH)
    lgb_part = {g: _z[f"lgb_{g}"] for g in pl.GROUP_COLS}
    mlp_part = {g: _z[f"mlp_{g}"] for g in pl.GROUP_COLS}
    print("저장된 예측 부분 로드:", PARTS_PATH.name, " (다시 만들려면 이 파일을 지우세요)")
else:
    lgb_part, mlp_part = build_final_parts()
    np.savez(PARTS_PATH, **{f"lgb_{g}": lgb_part[g] for g in pl.GROUP_COLS},
             **{f"mlp_{g}": mlp_part[g] for g in pl.GROUP_COLS})
    print("\n저장:", PARTS_PATH)

for g in pl.GROUP_COLS:
    print(f"  {g}: LGB {lgb_part[g].mean() / CAPACITY_KWH[g]:.4f} / "
          f"MLP {mlp_part[g].mean() / CAPACITY_KWH[g]:.4f}")

---

### 4-2. 제출 파일 생성 (학습 0회)

**블렌드 비중은 `0.5`로 확정됐다** (v5~v7 리더보드).

| 비중 | 0.3 | **0.5** | 0.7 |
|---|---|---|---|
| 리더보드 | 0.6413 | **0.6416** | 0.6405 |

곡선이 매우 완만해(0.6405~0.6416) 더 탐색할 값어치가 없다. **이 카드는 닫혔다.**

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 4-2. 제출 파일 생성 + 검증 + 직전 최고(v6)와 대조
# ══════════════════════════════════════════════════════════════════
MLP_BLEND_W = 0.5           # ⭐ v5~v7 리더보드로 확정
BEST_PATH = pl.REPO_ROOT / "submissions" / "20260803_v6_metricmlp05.csv"


def make_blend(w):
    pred = {g: np.clip((1 - w) * lgb_part[g] + w * mlp_part[g], 0, CAPACITY_KWH[g])
            for g in pl.GROUP_COLS}
    df = pd.DataFrame(pred, index=ctx.test.index)
    df["forecast_kst_dtm"] = ctx.test["kst_dtm"].dt.strftime("%Y-%m-%d %H:%M:%S")
    sub = build_submission(df, sample_path=SAMPLE_PATH)
    validate_submission(sub, sample_path=SAMPLE_PATH)
    return sub


submission = make_blend(MLP_BLEND_W)
print(f"✓ validate_submission() 통과 — {submission.shape}, MLP 비중 {MLP_BLEND_W}")

prev = pd.read_csv(BEST_PATH)
key = "forecast_kst_dtm" if "forecast_kst_dtm" in prev.columns else prev.columns[0]
assert prev[key].tolist() == submission[key].tolist(), "직전 제출과 시각/순서가 다름"

rows = []
for g in pl.GROUP_COLS:
    cap = CAPACITY_KWH[g]
    a, b = prev[g].to_numpy(float), submission[g].to_numpy(float)
    rows.append({"group": g, "v6 이용률": round(a.mean() / cap, 4),
                 "새 이용률": round(b.mean() / cap, 4),
                 "차이": round((b.mean() - a.mean()) / cap, 4),
                 "평균 절대변화(kWh)": round(np.abs(b - a).mean(), 1),
                 "상관(v6)": round(float(np.corrcoef(a, b)[0, 1]), 4)})
display(pd.DataFrame(rows).set_index("group"))

ok = [(pd.DataFrame(rows)["상관(v6)"] >= 0.98).all(),
      submission[pl.GROUP_COLS].notna().all().all(),
      (submission[pl.GROUP_COLS] >= 0).all().all(),
      all((submission[g] <= CAPACITY_KWH[g] + 1e-6).all() for g in pl.GROUP_COLS),
      len(submission) == 8760]
print(f"\n점검 5개: {['✓' if o else '✗' for o in ok]}  →  " +
      ("전부 통과" if all(ok) else "⚠️ 실패 항목 확인"))
print("""
⚠️ 20절 교훈: **로컬 편향이 +0.03을 넘는 구성은 로컬 점수를 믿지 말 것.**
   2025년은 예보 풍속이 3년치 어느 해보다 강한 해라 상향 편향의 여유가 없다.
   이용률이 v6보다 +0.02 이상 오르면 다시 생각할 것.""")

# ── 전부 ✓면 아래 주석을 풀어 저장 ────────────────────────────────
# print(save_submission(submission, "20260804_v8_설명.csv", sample_path=SAMPLE_PATH))

---

# 5. 로드맵 — 1등(0.67365)까지

## 격차의 정체
**-0.0321의 81%가 FICR**(0.4145 vs 0.4677)이고, 그 FICR 격차의 **98%가 "오차 10% 감소"** 하나로 설명된다.
1등은 산식 트릭을 쓰지 않았다. 그냥 더 정확하다. 그리고 **우리 σ의 89%는 예보 오차**다.

## 남은 카드

| 순위 | 카드 | 기대 | 비용 | 전제 |
|---|---|---|---|---|
| **1** | **2절 진단 → 대응 피처** (지형 speed-up / 대기 안정도 / 파워커브 기울기 손실) | **높음** | 각 12회 | 2절 판정 ★ |
| **2** | **격자 CNN 풍속 모델** | **높음** | 큼 | 2절이 전부 '무시'거나 1번 후 |
| 3 | 3절 MLP 튜닝 | 중 | 40분 | – |
| 4 | 발전량 LightGBM 하이퍼파라미터 | 낮 | 중 | 풍속 GBDT는 5방향 전부 실패했다 |
| 5 | 풍속 MLP(`WindMLP`) 튜닝 | 중 | 중 | 1·2번과 함께 |
| 6 | 2022년 학습 제외/가중 축소 | ? | 작음 | 20-2에서 정지율 이상 발견 |

## 값싼 물리 카드 (진단과 무관하게 해볼 만함)
- **파워커브 입력에 밀도보정 풍속** — IEC 61400-12-1은 밀도보정한 풍속을 파워커브에 넣으라고 한다.
  지금 `ws_est_corrected`를 만들어놓고 정작 파워커브에는 원본을 넣고 있다. **코드 한 줄**
- **TI = σ/U 정규화** — 현재 `roll7_std`는 분모(U)로 안 나눈 raw 변동폭이라 gain이 낮았다(0.037~0.231%).
  `roll7_std ÷ roll7_mean`이 바로 난류강도(TI)다

## ‼️ `train.ipynb` / `inference.ipynb` — 데드라인 2026-08-10
2차 평가 **필수 요건**. 성능이 아무리 좋아도 이게 없으면 무의미하다.
**`src/pipeline.py`가 생겨서 두 노트북은 얇은 껍데기면 된다** —
위 4-1의 `build_final_parts()`를 학습·저장 쪽과 로드·예측 쪽으로 쪼개면 된다(반나절).
`requirements.txt`도 이때 작성한다(**아직 없음**, `torch==2.13.0+cpu` 포함).